### RAG pipeline - data ingestion to vector db

In [42]:
# import required libraries
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HUGGING_FACE_API_KEY')

In [16]:
# Read all the pdf inside diractory
def load_all_pdf(path_to_pdfs):
    """Load and process all the pdfs"""
    all_documents = []
    path_dir = Path(path_to_pdfs)

    # load all pdf as list
    files = list(path_dir.glob('**/*.pdf'))

    if not files:
        return None
    print(f"Found {len(files)} pdfs.")
    
    for file in files:
        print(f"prpcessing {file}")
        try:
            loader = PyMuPDFLoader(str(file))
            documents = loader.load()
            print(f'found {len(documents)} pages\n')

            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error: {e}")
    
    return all_documents


# less control over the files. Instead return all documents at once
def load_all_pdf2(dir):
    loader = DirectoryLoader(
        dir,
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader
    )
    all_documents = loader.load()

    for doc in all_documents:
        doc.metadata['source_file'] = Path(doc.metadata['file_path']).name
        doc.metadata['file_type'] = 'pdf'
    
    return all_documents

documents = load_all_pdf('../data/pdf/ai-course')

Found 5 pdfs.
prpcessing ../data/pdf/ai-course/Chapter 3.pdf
found 63 pages

prpcessing ../data/pdf/ai-course/Chapter 1.pdf
found 20 pages

prpcessing ../data/pdf/ai-course/Chapter 4.pdf
found 11 pages

prpcessing ../data/pdf/ai-course/Chapter 5.pdf
found 15 pages

prpcessing ../data/pdf/ai-course/Chapter 6.pdf
found 16 pages



In [ ]:
# Split into chunks 
def split_text(docs, chunk_size = 1000, chunk_overlap = 200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=['\n\n', '\n', '. ', '! ', '? ', ' ', '']
    )
    chunk = splitter.split_documents(documents=docs)
    print(f'Splitted {len(docs)} documents into {len(chunk)} chunk')

    if chunk:
        print(f'{chunk[0].metadata}')
        print(f'{chunk[0].page_content[:200]}')

    return chunk

chunks = split_text(documents)

Splitted 125 documents into 122 chunk
{'producer': '', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/ai-course/Chapter 3.pdf', 'file_path': '../data/pdf/ai-course/Chapter 3.pdf', 'total_pages': 63, 'format': 'PDF 1.4', 'title': 'Chapter3_ProblemSolvingBySearching', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-05T03:07:03+06:00', 'trapped': '', 'modDate': "D:20260205030703+06'00'", 'creationDate': '', 'page': 0, 'source_file': 'Chapter 3.pdf', 'file_type': 'pdf'}



In [43]:
## Embedding and vector store
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma

hf_embeddings = HuggingFaceEndpointEmbeddings(  
    model="sentence-transformers/all-MiniLM-L6-v2",  
    task="feature-extraction",  
    huggingfacehub_api_token=HF_TOKEN,  
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    persist_directory='../chroma_db'
)

print("Vector store created successfully!")

Vector store created successfully!
